# Modeling Notebook

This notebook prepares user-level features, trains classification models, and selects the best model based on macro F1 score.

The notebook demonstrates a reproducible modeling pipeline from feature preparation through evaluation.

**Instructions:** Run the cells in order. Each cell includes comments explaining what it does and what output to expect.

In [14]:
# Cell 1: Import required libraries
# This cell loads all the Python packages needed for data processing, modeling, and evaluation.
# Run this first to ensure everything is available.

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [15]:
# Cell 2: Load the engineered user-level features
# This cell reads the user_features.csv file created in the feature engineering step.
# Expected output: A preview of the features DataFrame and the number of users.

base_dir = Path.cwd()
features = pd.read_csv(base_dir / 'user_features.csv')

print(f"Loaded {len(features)} user records with {len(features.columns)} features.")
print("Columns:", list(features.columns))
display(features.head())

Loaded 14 user records with 27 features.
Columns: ['user_id', 'total_transactions', 'total_transaction_volume', 'avg_transaction_amount', 'failed_transactions', 'deposit_amount', 'withdraw_amount', 'transfer_amount', 'payment_amount', 'airtime_amount', 'weekend_txn_count', 'business_txn_count', 'months_active', 'transactions_per_month', 'deposit_withdraw_ratio', 'weekend_ratio', 'weekend_activity', 'failed_ratio', 'business_usage', 'volume_norm', 'count_norm', 'avg_amount_norm', 'freq_norm', 'activity_score', 'activity_level', 'activity_threshold_low', 'activity_threshold_high']


,user_id,total_transactions,total_transaction_volume,avg_transaction_amount,failed_transactions,deposit_amount,withdraw_amount,transfer_amount,payment_amount,airtime_amount,...,failed_ratio,business_usage,volume_norm,count_norm,avg_amount_norm,freq_norm,activity_score,activity_level,activity_threshold_low,activity_threshold_high
0,msg_2026-03-20_190409,1028,5972090.0,5809.426070,26,155213.0,0.0,0.0,1224.0,1759548.0,...,0.025292,1,3.138208e-06,1.000000,2.062074e-07,1.000000,0.450001,high,0.041317,0.11618
1,msg_2026-03-21_032328,227,1645355.0,7248.259912,5,0.0,0.0,0.0,800.0,473794.0,...,0.022026,1,8.626647e-07,0.216243,2.659884e-07,0.346850,0.123431,high,0.041317,0.11618
2,msg_2026-03-21_185556,40,3389367.0,84734.175000,3,0.0,0.0,610000.0,2005.0,48144.0,...,0.075000,1,1.779886e-06,0.033268,3.485387e-06,0.167560,0.041830,medium,0.041317,0.11618
3,msg_2026-03-21_195936,153,1922611.0,12566.084967,7,15130.0,0.0,0.0,925.0,703961.0,...,0.045752,1,1.008481e-06,0.143836,4.869343e-07,0.351039,0.106167,medium,0.041317,0.11618
4,msg_2026-03-22_213925,51,810073.0,15883.784314,0,0.0,0.0,0.0,0.0,33373.0,...,0.000000,1,4.233680e-07,0.044031,6.247786e-07,0.171582,0.045324,medium,0.041317,0.11618


In [16]:
# Cell 3: Prepare features and target for modeling
# This cell separates the target variable (activity_level) from the predictors.
# It also identifies numeric and categorical columns for preprocessing.
# Expected output: Summary of data shapes and column types.

target = 'activity_level'

# Remove columns that should not be used as predictors.
drop_cols = {'user_id', 'activity_threshold_low', 'activity_threshold_high'}
x = features.drop(columns=[target] + [col for col in drop_cols if col in features.columns])
y = features[target]

# Separate numeric and categorical inputs for preprocessing.
numeric_cols = x.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in x.columns if col not in numeric_cols]

print(f"Target variable: {target}")
print(f"Features: {len(x.columns)} columns ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
print(f"Classes: {sorted(y.unique())}")
print(f"Class distribution:\n{y.value_counts()}")

Target variable: activity_level
Features: 23 columns (23 numeric, 0 categorical)
Classes: ['high', 'low', 'medium']
Class distribution:
activity_level
high      5
low       5
medium    4
Name: count, dtype: int64


In [17]:
# Cell 4: Set up data preprocessing pipeline
# This cell creates a preprocessor that scales numeric features and encodes categorical ones.
# Expected output: Confirmation that the preprocessor is configured.

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ]
)

print("Preprocessor configured:")
print("- Numeric features will be standardized (mean=0, std=1)")
print("- Categorical features will be one-hot encoded")
print(f"Total features after preprocessing: {len(numeric_cols) + len(categorical_cols) * len(features[categorical_cols].nunique()) if categorical_cols else len(numeric_cols)}")

Preprocessor configured:
- Numeric features will be standardized (mean=0, std=1)
- Categorical features will be one-hot encoded
Total features after preprocessing: 23


In [18]:
# Cell 5: Split data into training and testing sets
# This cell uses stratified splitting to preserve class balance in train/test sets.
# Expected output: Summary of split sizes and class distributions.

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

print(f"Training set: {len(x_train)} samples")
print(f"Testing set: {len(x_test)} samples")
print(f"Training class distribution:\n{y_train.value_counts()}")
print(f"Testing class distribution:\n{y_test.value_counts()}")

Training set: 11 samples
Testing set: 3 samples
Training class distribution:
activity_level
high      4
low       4
medium    3
Name: count, dtype: int64
Testing class distribution:
activity_level
high      1
medium    1
low       1
Name: count, dtype: int64


In [19]:
# Cell 6: Define the models to compare
# This cell sets up three classifiers with balanced class weights to handle imbalanced data.
# Expected output: List of models being evaluated.

models = {
    'logistic_regression': LogisticRegression(max_iter=200, class_weight='balanced'),
    'decision_tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
}

print("Models to evaluate:")
for name, model in models.items():
    print(f"- {name}: {model.__class__.__name__}")

Models to evaluate:
- logistic_regression: LogisticRegression
- decision_tree: DecisionTreeClassifier
- random_forest: RandomForestClassifier


In [20]:
# Cell 7: Train and evaluate models
# This cell trains each model on the training data and evaluates on the test set.
# Expected output: A table comparing model performance metrics.

results = []
best_name = None
best_f1 = -1
best_model = None

for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(x_train, y_train)
    preds = pipeline.predict(x_test)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision_macro': precision_score(y_test, preds, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, preds, average='macro', zero_division=0),
        'f1_macro': f1,
    })
    if f1 > best_f1:
        best_f1 = f1
        best_name = name
        best_model = pipeline

results_df = pd.DataFrame(results)
results_df.to_csv('model_comparison.csv', index=False)
print('Model comparison results saved to model_comparison.csv')

display(results_df)

Model comparison results saved to model_comparison.csv


,model,accuracy,precision_macro,recall_macro,f1_macro
0,logistic_regression,0.000000,0.0,0.000000,0.000000
1,decision_tree,0.666667,0.5,0.666667,0.555556
2,random_forest,0.666667,0.5,0.666667,0.555556


In [ ]:
# Cell 8: Save the best model and show detailed results
# This cell saves the best-performing model and displays a detailed classification report.
# Expected output: Confirmation of saved model and detailed performance breakdown.

if best_model is not None:
    joblib.dump(best_model, 'best_model.pkl')
    print(f'Best model: {best_name} with macro F1 = {best_f1:.4f}')
    best_preds = best_model.predict(x_test)
    print('\nDetailed classification report for the best model:')
    print(classification_report(y_test, best_preds, zero_division=0))
else:
    print("No best model found.")

Best model: decision_tree with macro F1 = 0.5556

Detailed classification report for the best model:
              precision    recall  f1-score   support

        high       1.00      1.00      1.00         1
         low       0.50      1.00      0.67         1
      medium       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.50      0.67      0.56         3
weighted avg       0.50      0.67      0.56         3

